# 12 Final Conclusion And Best Model

This is the final reporting notebook for the active hourly DA pipeline.

It does three things:
- load the latest combined `FS2` comparison run
- select the final challenger on **validation only**
- display the held-out test results and objective case-week plots for the selected winner

Prerequisite:
- run `11_fs2_benchmark_and_shortlisting.ipynb` first so a fresh `model_comparison` run exists
- refresh `visual_case_weeks` at the end of this notebook if those artifacts are missing or stale


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys
import time

import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents] if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)
PACKAGE_ROOT = REPO_ROOT / "scripts/Data/02_Forecasting/01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from hourly_da.core.config import HourlyDAPipelineConfig
from hourly_da.core.external_features import build_external_family_catalog, load_external_feature_store
from hourly_da.core.methodology import (
    STARTER_ENDOGENOUS_FEATURE_NOTE,
    feature_stage_policy_frame,
    model_status_frame,
    shortlisting_policy_frame,
)
from hourly_da.core.reporting import find_latest_run, load_csv, load_json
from hourly_da.core.tuning import build_tuning_placeholder, tuning_cadence_frame, tuning_snippet_frame
from hourly_da.models import naive_model_names
from hourly_da.notebook_support import estimate_run_duration_seconds, format_duration, load_selected_case_weeks

config = HourlyDAPipelineConfig(
    input_csv=REPO_ROOT / "data/01_cleaned/Day_ahead_prices/DA_prices/hourly/da_prices_all_regions_hourly.csv",
    raw_root=REPO_ROOT / "data/00_Raw/DA_Prices",
    cleaned_feature_root=REPO_ROOT / "data/01_cleaned",
    output_root=REPO_ROOT / "data/02_Forecasting/01_DA_prices/hourly_da",
)
output_root = config.output_root


def latest_run_or_none(run_label: str) -> Path | None:
    try:
        return find_latest_run(output_root, run_label)
    except FileNotFoundError:
        return None


def ensure_required_modules(module_names: list[str], install_command: str | None = None) -> None:
    missing = [module_name for module_name in module_names if importlib.util.find_spec(module_name) is None]
    if not missing:
        return

    message_lines = [
        "Missing required package(s) in the active notebook interpreter: " + ", ".join(missing),
        f"Active interpreter: {sys.executable}",
    ]
    if install_command:
        message_lines.append(f"Install command: {install_command}")
    raise RuntimeError("\n".join(message_lines))


def run_command_with_live_output(command: list[str]) -> None:
    print("Running command:")
    print(" ".join(str(part) for part in command))
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )

    output_tail: list[str] = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        output_tail.append(line)
        if len(output_tail) > 40:
            output_tail.pop(0)

    return_code = process.wait()
    if return_code != 0:
        tail_text = "".join(output_tail).strip()
        message = f"Command failed with exit code {return_code}."
        if tail_text:
            message += "\nLast output:\n" + tail_text
        raise RuntimeError(message)


## Optional refresh hook


In [ ]:
ALLOW_HEAVY_RERUN = False

if ALLOW_HEAVY_RERUN:

    estimate = estimate_run_duration_seconds(output_root, "visual_case_weeks")
    if estimate is not None:
        print(
            "Heavy rerun warning: latest comparable run "
            f"{estimate['run_id']} suggests about {format_duration(float(estimate['estimate_seconds']))}."
        )
    else:
        print("Heavy rerun warning: no comparable runtime estimate was found for this stage.")

    command = [sys.executable, str(PACKAGE_ROOT / "run_visual_case_weeks.py")]

    started = time.perf_counter()
    run_command_with_live_output(command)
    elapsed_seconds = time.perf_counter() - started
    print(f"Actual wall-clock time: {format_duration(elapsed_seconds)}")
else:
    print("Rerun skipped. Set ALLOW_HEAVY_RERUN = True only when you are ready to execute the finalized pipeline.")


In [ ]:
from hourly_da.notebook_support import (
    build_compared_models_overview,
    build_reporting_summary_table,
    load_standard_report_bundle,
    render_plot_gallery,
    style_model_overview_table,
    style_reporting_summary_table,
)


In [ ]:
rows = []
for run_label in ['model_comparison', 'visual_case_weeks', 'case_week_selection']:
    run_dir = latest_run_or_none(run_label)
    rows.append(
        {
            "run_label": run_label,
            "latest_run": str(run_dir) if run_dir is not None else "not yet run",
        }
    )

display(pd.DataFrame(rows))


## Combined comparison summary

The final challenger is chosen on the validation split using the full-horizon reporting level. The test split is shown only after that choice is frozen.


In [ ]:
comparison_run_dir = latest_run_or_none("model_comparison")
visual_run_dir = latest_run_or_none("visual_case_weeks")

if comparison_run_dir is None:
    print("No combined comparison run exists yet. Run 11_fs2_benchmark_and_shortlisting.ipynb first.")
else:
    print(f"Comparison run: {comparison_run_dir}")
    if visual_run_dir is None:
        print("No visual_case_weeks run exists yet. Use the optional refresh hook at the end of this notebook.")
    else:
        print(f"Visual case week run: {visual_run_dir}")

    active_families = (
        model_status_frame()
        .loc[lambda df: df["status"] == "Active", "model_family"]
        .astype(str)
        .tolist()
    )
    active_challenger_families = ["lear", "xgboost", "prophet"]

    bundle = load_standard_report_bundle(output_root=output_root, current_run_dir=comparison_run_dir)
    comparison_specs = bundle["comparison_specs"].copy()
    comparison_specs = comparison_specs[comparison_specs["model_family"].isin(active_families)].copy()
    comparison_specs = comparison_specs.sort_values(["comparison_order", "display_name"]).reset_index(drop=True)

    official_naive = bundle["official_naive"]
    official_naive_model = str(official_naive["model"])
    model_order = comparison_specs["display_name"].astype(str).tolist()
    model_name_order = comparison_specs["model"].astype(str).tolist()

    overview = build_compared_models_overview(comparison_specs, official_naive_model=official_naive_model)
    display(style_model_overview_table(overview))

    active_metrics = bundle["metrics_by_reporting_level"][
        bundle["metrics_by_reporting_level"]["model"].isin(model_name_order)
    ].copy()

    validation_summary = build_reporting_summary_table(
        metrics_by_reporting_level=active_metrics,
        split_name="validation",
        model_order=model_order,
    )
    test_summary = build_reporting_summary_table(
        metrics_by_reporting_level=active_metrics,
        split_name="test",
        model_order=model_order,
    )
    display(style_reporting_summary_table(validation_summary, caption="Validation summary"))
    display(style_reporting_summary_table(test_summary, caption="Test summary"))

    selection_frame = active_metrics[
        (active_metrics["dataset_split"] == "validation")
        & (active_metrics["reporting_level"] == "stitched_all_horizon")
        & (active_metrics["model_family"].isin(active_challenger_families))
    ].copy()

    if selection_frame.empty:
        print("No active challenger models were found in the current comparison run.")
    else:
        selection_frame["abs_bias"] = selection_frame["bias"].astype(float).abs()
        selection_frame = selection_frame.sort_values(["mae", "rmse", "abs_bias", "comparison_order", "model"])
        validation_row = selection_frame.iloc[0]
        selected_model = str(validation_row["model"])
        selected_display = str(validation_row.get("display_name", selected_model))

        test_row = active_metrics[
            (active_metrics["dataset_split"] == "test")
            & (active_metrics["reporting_level"] == "stitched_all_horizon")
            & (active_metrics["model"].astype(str) == selected_model)
        ].copy()
        test_row = test_row.sort_values(["mae", "model"]).head(1)

        conclusion_row = {
            "selected_final_model": selected_display,
            "selected_internal_model": selected_model,
            "naive_benchmark": official_naive_model,
            "validation_mae": float(validation_row["mae"]),
            "validation_rmae": float(validation_row["rmae_vs_official_naive"]),
            "test_mae": float(test_row.iloc[0]["mae"]) if not test_row.empty else float("nan"),
            "test_rmae": float(test_row.iloc[0]["rmae_vs_official_naive"]) if not test_row.empty else float("nan"),
            "test_rmse": float(test_row.iloc[0]["rmse"]) if not test_row.empty else float("nan"),
            "test_bias": float(test_row.iloc[0]["bias"]) if not test_row.empty else float("nan"),
        }

        display(
            Markdown(
                f"### Selected final challenger\n"
                f"`{selected_display}` is selected on **validation** full-horizon MAE. "
                f"The test rows below are shown only after that selection is frozen."
            )
        )
        display(pd.DataFrame([conclusion_row]))

        dm_rows = bundle["diebold_mariano_by_reporting_level"].copy()
        dm_rows = dm_rows[
            (dm_rows["challenger_model"].astype(str) == selected_model)
            & (dm_rows["benchmark_model"].astype(str) == official_naive_model)
        ].copy()
        if "reporting_level" in dm_rows.columns:
            dm_rows = dm_rows[dm_rows["reporting_level"].astype(str) != "guidance_only"].copy()
        if dm_rows.empty:
            print("No Diebold-Mariano rows were found for the selected final challenger.")
        else:
            display(
                dm_rows[
                    [
                        "dataset_split",
                        "reporting_level_label",
                        "n_obs",
                        "dm_stat",
                        "p_value",
                    ]
                ]
                .sort_values(["dataset_split", "reporting_level_label"])
                .reset_index(drop=True)
            )


## Objective case weeks

Only the required thesis cases are displayed here:
- typical winter
- typical summer
- high volatility


In [ ]:
required_categories = ["typical_winter", "typical_summer", "high_volatility"]

if visual_run_dir is None:
    print("No visual_case_weeks run exists yet.")
else:
    selected_weeks = load_csv(visual_run_dir, "selected_weeks.csv")
    week_metrics = load_csv(visual_run_dir, "week_metrics.csv")

    if "category" in selected_weeks.columns:
        selected_weeks = selected_weeks[selected_weeks["category"].isin(required_categories)].copy()
    if "category" in week_metrics.columns:
        week_metrics = week_metrics[week_metrics["category"].isin(required_categories)].copy()

    display(selected_weeks)

    if "selected_model" in locals():
        focus_models = [official_naive_model, selected_model]
        focus_model_order = {model_name: position for position, model_name in enumerate(focus_models)}
        focus_week_metrics = week_metrics[
            week_metrics["model"].isin(focus_models)
        ].copy()
        display(
            focus_week_metrics
            .assign(_model_order=lambda frame: frame["model"].map(focus_model_order).fillna(len(focus_model_order)))
            .sort_values(["category", "_model_order", "model"])
            .drop(columns=["_model_order"])
            .reset_index(drop=True)
        )
    else:
        full_model_order = {model_name: position for position, model_name in enumerate(model_name_order)}
        display(
            week_metrics
            .assign(_model_order=lambda frame: frame["model"].map(full_model_order).fillna(len(full_model_order)))
            .sort_values(["category", "_model_order", "model"])
            .drop(columns=["_model_order"])
            .reset_index(drop=True)
        )

    actual_plots = [
        visual_run_dir / "plots" / f"{category}_actual_only.png"
        for category in required_categories
        if (visual_run_dir / "plots" / f"{category}_actual_only.png").exists()
    ]
    overlay_plots = [
        visual_run_dir / "plots" / f"{category}_actual_vs_all_models.png"
        for category in required_categories
        if (visual_run_dir / "plots" / f"{category}_actual_vs_all_models.png").exists()
    ]

    display(Markdown("### Actual price weeks"))
    display(render_plot_gallery(actual_plots, columns=3))
    display(Markdown("### D-only model overlays"))
    display(render_plot_gallery(overlay_plots, columns=1))
